# Model Experiments: Kryptonite-20, -30 and -40

The paper makes three claims about Kryptonite-n that this notebook tests:

1. Polynomial logistic regression stays below 59% up to degree 10, so polynomial basis expansion fails ("a counterexample to universal approximation").
2. Quadrupling the training data does not help.
3. Nothing reaches the 90% that counts as solving the dataset: even KryptoNet-40, an MLP pretrained on 200,000 examples of a related task and used frozen, scores worse than a coin flip.

The paper's Future Works suggests networks trained from scratch and tree-based models, and asks what degree and how many examples a polynomial would need.

KryptoNet-40 on Kryptonite-40-Adapt is the LoRA notebook's part. This notebook also picks one final pipeline per dataset, scores it on a locked test set and writes the hidden-set predictions to `hiddenlabels/`.

The helper code is in `experiment_tools.py` (needs `codecarbon`). Results are cached in `exp_cache/` (git-ignored) so an interrupted run resumes; delete it for a fresh run. `KRYP_QUICK=1` runs a dry run on 3,000 rows whose numbers mean nothing.

In [ ]:
import experiment_tools as et
from experiment_tools import G, NS, cv_by_fold, cv_config

print(
    "QUICK dry run" if et.QUICK else "Full run",
    f"- {et.K}-fold CV on 80% of the rows, 20% locked test",
)

## Data and Protocol

A stratified 20% of the labelled rows is locked away until the end. The other 80% (dev) is split into five stratified folds; every choice, including the bit weights of Experiment 3, is made on the training folds only. Accuracies are mean ± sd over the folds; two models are compared fold by fold. The locked-test accuracy has the binomial 95% interval p ± 1.96·√(p(1−p)/N).

The inputs are ±1 bits plus Gaussian noise, so every model uses the sign of each input, except the paper's baseline, which uses the raw inputs as in the paper.

In [ ]:
et.check_protocol()

## Experiment 1: Higher Degree on All Bits

*Claim 1.* *Hypothesis:* a higher degree does not help on all n bits, because the number of monomials grows far faster than the number of training rows.

Logistic regression on explicit monomials (products of distinct bits) with an L2 penalty, `C` chosen by CV, up to the largest degree that fits in memory (4, 3, 2 for n = 20, 30, 40). Beyond that, the paper's own polynomial kernel (`C = 0.85`, plus an intercept) at degrees 6, 10, 15, n and 2n. From degree n on, the kernel can represent every function of the bits. If it then fits the training rows perfectly but not the validation rows, the failure is one of generalisation, not of approximation.

In [ ]:
et.test_kernel()
for n in NS:
    cv_config(n, "BASE", "paper", "all", "deg2", dict(deg=2, C=0.85))
    for d in range(1, G["MAX_DEG"][n] + 1):
        for C in G["C_L2"]:
            cv_config(n, "LR", "lr_l2", "all", f"deg{d}", dict(deg=d, C=C))
    for d in G["KP_DEGS"][n]:
        cv_config(n, "KPOLY", "kpoly", "all", f"deg{d}", dict(deg=d, C=G["KP_C"]))
et.table_degree_all()

## Experiment 2: Models Trained From Scratch on All Bits

*Claim 3.* The two model types the paper's Future Works suggests, tuned by the same CV on all n bits:

- a random forest;
- an MLP with KryptoNet-40's architecture (four hidden layers of 512), trained from scratch on Kryptonite-n with Adam and early stopping. The paper only used this architecture pretrained on Kryptonite-40-Pre and frozen.

In [ ]:
for n in NS:
    for f in G["RF_MAXF"]:
        for leaf in G["RF_LEAF"]:
            cv_config(n, "RF", "rf", "all", "rf", dict(max_features=f, msl=leaf))
    if et.RUN_MLP:
        cv_by_fold(n, "mlp", dict(h=G["MLP_H"]), "4x512")
et.table_families()

## Experiment 3: The Polynomial Kernel With Forest-Weighted Bits

*Claim 1, and the paper's question of what a polynomial needs.* Experiment 1 penalises every monomial equally, as the paper does. The forest of Experiment 2 beats it on the same bits, and a forest only splits on bits that help it. *Hypothesis:* the label depends on a few of the n bits, and the polynomial fails because almost all its monomials involve the others, not because its degree is too low.

This is a two-stage method, not the paper's estimator:

1. In each training fold, a random forest is fitted on three quarters of the training rows. Each bit's weight wᵢ is its permutation importance on the remaining quarter (Breiman, 2001): the drop in the forest's accuracy when that bit is shuffled. Negative importances are set to 0 and the largest weight is scaled to 1.
2. Logistic regression on all n bits with the polynomial kernel

$$k(x, x') = \left(\frac{1 + \sum_i w_i x_i x'_i}{1 + \sum_i w_i}\right)^d$$

The result is still a degree-d polynomial of the bits. The weights change only how strongly each monomial is penalised: monomials of low-weight bits are penalised more heavily, and a monomial containing a zero-weight bit is left out. Giving each input its own scale in a kernel is the idea behind automatic relevance determination (MacKay, 1994; Neal, 1996). The same kernel with every weight 1 is the comparison.

In [ ]:
et.table_weights()
et.fig_weights()
for n in NS:
    for d in G["WK_DEGS"]:
        cv_config(n, "UK", "ukpoly", "all", f"deg{d}", dict(deg=d, C=G["WK_UNIFORM_C"]))
        for C in G["WK_C"]:
            cv_config(n, "WK", "wkpoly", "all", f"deg{d}", dict(deg=d, C=C))
et.table_weighted_kernel()
et.fig_degree()

## Experiment 4: Amount of Training Data

*Claim 2, and the paper's question of how many examples are needed.* The best model of each kind is trained on nested random subsets of the training folds (first three folds) and scored on the held-out fold; the weighted kernel computes its weights on the subset. For LR and the kernels, `C` is scaled with the number of rows so the penalty per row stays fixed.

In [ ]:
et.fig_learning_curves(et.run_learning_curves())

## Final Pipeline and Hidden-Set Predictions

The final pipeline per n is the candidate with the highest CV mean. It is refitted on all dev rows (its bit weights included) and scored once on the locked rows. Two controls must score about 0.5: shuffled training labels (otherwise something leaks) and, for the weighted kernel, the same weights moved to the wrong bits, the largest on the least important (otherwise the weights are not what makes it work). `check_caps` lists any fit that stopped at its iteration limit.

For the hidden set the pipeline is refitted on all labelled rows. As the hidden labels are unknown, its predictions must agree with those of the best model of another kind that scores above 0.9 in CV: two models with error rates e₁ and e₂ can disagree on at most e₁ + e₂ of the rows, so a lower agreement (with 2 points of slack) stops the notebook. Then `hiddenlabels/y_predicted_{20,30,40}.npy` are written and checked with the course's `check_submission.py`.

In [ ]:
FINAL = et.choose_final()
et.run_controls(FINAL)
et.table_final(FINAL)
et.check_caps()
print()
et.predict_hidden(FINAL)

## Summary

In [ ]:
et.fig_summary_table(FINAL)

## Energy and Emissions

CPU energy measured with CodeCarbon while each result was computed, from the processor's own energy counters (the whole package, so other programs should be closed). Every model runs on the CPU only, so no GPU energy is counted; RAM and screen are not measured. If the CPU counters cannot be read, nothing is reported. Emissions use the GB grid's 2024 average, 125 g CO2 per kWh (NESO, *Britain's Electricity Explained: 2024 Review*).

In [ ]:
et.carbon_report()